# Preparação e análise de Dados em Python #

Nesta aula, aprenderemos um pouco a como entender melhor um conjunto de dados. Nos tempos atuais, é cada vez mais fácil conseguir quantidades cada vez maiores de dados. Contudo, isto vem com um custo: quanto mais dados, mais difícil é de __entender__ os mesmos.

Hoje, aprenderemos a utilizar a biblioteca Pandas para manipular dados.

In [ ]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
#!pip3 install pandas --user

Na aula de hoje, usaremos uma base de dados sobre número de vendas de jogos de videogame até 2016. Nosso objetivo é entender melhor os dados para na sequência produzir modelos de regressão que melhor prevejam os números de vendas. Baixe o arquivo que está no Colabweb e coloque-o descomprimido na mesma pasta em que está rodando este notebook:

In [ ]:
vg = pd.read_csv('Video_Games_Sales_as_at_22_Dec_2016.csv')

O Pandas é uma biblioteca de manipulação de dados. A função read_csv lê __todo__ conteúdo do arquivo e coloca em uma estrutura de dados. Existem funções similares para diversos tipos de arquivos, como `xls` e `json`.

Agora vamos visualizar o conteúdo dos nossos dados. Para isto, usaremos a função head, que mostra os 5 primeiros registros:

In [ ]:
#vg.head()
#vg.tail()
vg.sample(5)

In [ ]:
vg.columns

In [ ]:
vg.info()

Acessando os dados de um jogo especifico:

In [ ]:
#vg.loc[130]
#vg2 = vg.loc[[1,5,10,20]]
vg.loc[0:100]

Podemos também acessar colunas especificas através dos seus nomes:

In [ ]:
#vg[ ['Name','Critic_Score'] ]

vg.Global_Sales

Quantos jogos para cada sistema nós temos?

In [ ]:
vg.Genre.value_counts().plot(kind='bar')


Contando a quantidade de valores diferentes de um atributo numérico:

In [ ]:
vg.Critic_Score.value_counts()

Muito né? melhor seria colocar em categorias:

In [ ]:
b = list(range(0,101,10))
#b
vg.Critic_Score.value_counts(bins=b)


O Pandas oferece vários métodos para entender melhor os dados com estatísticas básicas:

In [ ]:
vg.describe()

Epa! Tem um jogo com lançamento em 2020! Não pode, pois a base é de jogos lançados até 2016. Na verdade, isso é super comum com dados reais, de haver sujeira na base de dados.

E agora, como podemos identificar que jogo é esse ?

In [ ]:
vg[vg.Year_of_Release==2020]
#vg.loc[vg.Year_of_Release==2020, ['Year_of_Release']] = [2010]

Conseguiu entender como funcionou? Brinque um pouco com os dados para entender melhor.

Quais são os jogos mais recentes da base para Nintendo Wii?

In [ ]:
vg[vg.Platform=='PS2'].sort_values(['Global_Sales','Year_of_Release'],ascending = False).head()

A saída de boa parte de comandos em Pandas gera objetos do tipo dataframe ou series. Estes objetos novos podem ser utilizados com outras funções pandas, em cascata.

Isto são condições compostas

Jogos de PS3 que tiveram mais do que 20 críticas ordenados pela nota da critica:

In [ ]:
#vg[(vg.Platform=='PS2')][(vg.Critic_Count>20)].sort_values(['Critic_Score']).head()
vg[(vg.Platform=='PS2') & (vg.Critic_Count>20)].sort_values(['Critic_Score']).head()

Exercício 1:


1 - Gere uma base vg2, contendo todos os jogos lançados de 2000 a 2016 e a descreva com describe

2 - Mostre todos os jogos lançados até o ano 2000 onde as vendas japonesas foram acima das americanas, ordenado por vendas globais

3 - Conte, para cada plataforma, quantos jogos com nota de usuário acima de 8 cada uma tem

### Usando pandas para plotar com pyplot ###

Uma coisa legal no pandas é que ele é integrado ao pyplot/matplotlib.

Exemplo: Qual a relação entre as vendas globais e a nota da crítica?

In [ ]:
vg[vg.Global_Sales<80].plot(kind='scatter',x='Critic_Score',y='Global_Sales')
#outlier
#vg[vg.Global_Sales>80]

Os valores muito altos (como o Wii Sports com 80 milhões de cópias), estão dificultando a visualização. Para resolver este problema, podemos ou eliminar estes valores do plot ou plotar em escala logaritmica:

In [ ]:
#vg[vg.Global_Sales<20].plot(kind='scatter',x='Critic_Score',y='Global_Sales')

vg.plot(kind='scatter',x='Critic_Score',y='Global_Sales',logy=True)

### Outras correlações mais sofisticadas ###

Como será a comparação entre as notas dos críticos para RPGs em comparação com jogos de corrida?

In [ ]:
# Primeiro veremos o histograma global
vg.Critic_Score.hist(bins = 20)

In [ ]:
#Agora apenas dos RPGs
vg[vg.Genre=='Role-Playing'].Critic_Score.hist(bins = 20)

In [ ]:
# Podemos comparar os RPGs com os jogos de corrida

bins = list(range(0,101,5))
vgval = vg[vg.Critic_Score.notnull()] # Remover os jogos sem valor de Critic_Score

plt.hist(vgval[vgval.Genre=='Role-Playing'].Critic_Score,bins,alpha=0.5,label='RPG')
plt.hist(vgval[vgval.Genre=='Racing'].Critic_Score,bins,alpha=0.5,label='Corrida')
plt.legend()


E se eu quiser comparar variáveis categóricas?

Vamos analisar duas: Gênero e Plataforma.

In [ ]:
pd.crosstab(vg.Platform,vg.Genre,margins = True)

E se a gente quisesse esses valores em porcentagens? Podemos então aplicar uma __função lambda__ nos dados:

In [ ]:
pd.crosstab(vg.Platform,vg.Genre,margins = True,normalize='index')#.apply(lambda v:100*v/v[-1],axis = 1)

E se eu quiser uma tabela dessas, mas apenas contendo os jogos que venderam mais de 10 milhões de unidades?

In [ ]:
pd.crosstab(vg[vg.Global_Sales>10].Platform,vg[vg.Global_Sales>10].Genre,margins = True)#,normalize='index')

## Tabelas Pivo ##

Muitas vezes, queremos cruzar dados de formas mais complexas. A tabela pivo ajuda muito nisso.


Exemplo: Quem teve os melhores RPGs e adventures de acordo com a crítica? O primeiro ou o segundo Playstation?

In [ ]:
rpgs = vg[(vg.Platform.isin(['PS2','PS']))& vg.Genre.isin(['Role-Playing','Adventure']) ]
#rpgs.head()
pivot = rpgs.pivot_table(values=['Critic_Score'],index = ['Platform','Genre'],aggfunc = ["mean",])
pivot

In [ ]:
pivot.plot(kind='barh')

Exercício 2:

Faça uma tabela contendo a media das notas de usuários dos jogos das Publishers Capcom e Activision de acordo com o Rating

E se eu quiser saber quantas unidades o jogo mais vendido de cada plataforma vendeu?

In [ ]:
#quantas unidades o jogo mais vendido de cada plataforma vendeu?
plats = vg[(vg.Platform.isin(['PS3','X360']))]

plats.pivot_table(values=['Global_Sales'],index = ['Platform','Genre'],aggfunc = ["max",])



## Agrupando ##

Você pode usar o comando groupby para gerar novas bases de dados modificadas. Por exemplo, você pode tentar prever as vendas totais de plataformas ao invés de jogos individuais. Contudo, para você utilizar os dados agrupados, você deve usar uma função de agregação (sum,count, min,max, mean, etc).

### Exemplo ###

Criando um agrupamento por plataforma

In [ ]:
vgplat = vg.groupby(['Platform'])
vgplat.Global_Sales.sum()
vgplat.Critic_Score.min()

Fazendo uma relação para descobrir total de vendas de um console pelo total de jogos lançados para ele:

In [ ]:
x = vgplat.Name.count()
y = vgplat.Global_Sales.sum()
plt.scatter(x,y)

In [ ]:
#Total de anos que um console teve jogos

anos = vgplat.Year_of_Release.max() - vgplat.Year_of_Release.min()
anos.plot(kind='bar')

Vocês poderiam usar a informação de ano para ponderar alguma outra informação usada na regressão por exemplo. Contudo, tem algo estranho né? O Nintendo DS está com 35 anos de jogos lançados. Vocês conseguem descobrir o porquê?